In [1]:
import json
import re
import random
import spacy
from spacy.tokens import DocBin

In [2]:
# Open the file in read mode
with open('train.json', 'r') as file:
    dataset = json.load(file)

In [3]:
dataset[0:3]

[{'text': "Abhishek Jha Application Development Associate - Accenture  Bengaluru, Karnataka - Email me on Indeed: indeed.com/r/Abhishek-Jha/10e7a8cb732bc43a  • To work for an organization which provides me the opportunity to improve my skills and knowledge for my individual and company's growth in best possible ways.  Willing to relocate to: Bangalore, Karnataka  WORK EXPERIENCE  Application Development Associate  Accenture -  November 2017 to Present  Role: Currently working on Chat-bot. Developing Backend Oracle PeopleSoft Queries for the Bot which will be triggered based on given input. Also, Training the bot for different possible utterances (Both positive and negative), which will be given as input by the user.  EDUCATION  B.E in Information science and engineering  B.v.b college of engineering and technology -  Hubli, Karnataka  August 2013 to June 2017  12th in Mathematics  Woodbine modern school  April 2011 to March 2013  10th  Kendriya Vidyalaya  April 2001 to March 2011  SKIL

In [4]:
# 1. Load a blank English spaCy model for tokenization
nlp = spacy.blank("en")

In [5]:
def clean_text(text): #data cleaning
    """
    Cleans structural noise and URLs while preserving exact 
    character counts for index alignment.
    """
    # urls = re.findall(r'https?://\S+|www\.\S+', text)
    # for url in urls:
    #     text = text.replace(url, " " * len(url)) 
    text = re.sub(r'[\t\r\n]', ' ', text)
    text = re.sub(r'[●•❖➣]', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9 ]', '', text)
    
    return text

In [6]:
valid_entities = [] # global variable

In [7]:
def convert_to_docbin(dataset):
    """
    Cleans text, filters entities, aligns indices to tokens,
    and packages data into a spaCy DocBin container.
    """
    db = DocBin()

    for record in dataset:
        text = record.get("text", "")
        annotations = record.get("annotations", [])

        cleaned_text = clean_text(text)
        doc = nlp.make_doc(cleaned_text)

        # IMPORTANT: initialize this for every document
        valid_entities = []

        # Sort annotations by start position
        sorted_annotations = sorted(annotations, key=lambda x: x[0])
        last_end_offset = -1

        for start, end, label in sorted_annotations:

            entity_string = cleaned_text[start:end]
            stripped_string = entity_string.strip()

            # Recalculate offsets after stripping whitespace
            if len(stripped_string) != len(entity_string):
                try:
                    start_offset = entity_string.index(stripped_string)
                    start = start + start_offset
                    end = start + len(stripped_string)
                except ValueError:
                    continue

            # Validate boundaries
            if start < 0 or end > len(cleaned_text) or start >= end:
                continue

            # Drop overlapping entities
            if start < last_end_offset:
                continue

            # Align entity to spaCy token boundaries
            span = doc.char_span(
                start,
                end,
                label=label.upper(),
                alignment_mode="contract"
            )

            if span is not None:
                valid_entities.append(span)
                last_end_offset = end

        # Assign entities
        doc.ents = valid_entities

        # Add document to DocBin
        db.add(doc)

    return db

In [8]:
# --------------------------------------------------
# 1. Shuffle the dataset
# --------------------------------------------------

random.seed(42)
random.shuffle(dataset)


# --------------------------------------------------
# 2. Calculate split sizes
# --------------------------------------------------

total = len(dataset)

train_size = int(0.80 * total)
val_size = int(0.10 * total)

train_data = dataset[:train_size]
val_data = dataset[train_size:train_size + val_size]
test_data = dataset[train_size + val_size:]


# --------------------------------------------------
# 3. Convert each split into a DocBin
# --------------------------------------------------

train_db = convert_to_docbin(train_data)
val_db = convert_to_docbin(val_data)
test_db = convert_to_docbin(test_data)


# --------------------------------------------------
# 4. Save the DocBins
# --------------------------------------------------

train_db.to_disk("train.spacy")
val_db.to_disk("validation.spacy")
test_db.to_disk("test.spacy")


# --------------------------------------------------
# 5. Check the sizes
# --------------------------------------------------

print(f"Total records:      {total}")
print(f"Training records:   {len(train_data)}")
print(f"Validation records: {len(val_data)}")
print(f"Test records:       {len(test_data)}")

Total records:      5960
Training records:   4768
Validation records: 596
Test records:       596
